In [1]:
# Cell 1: Import libraries and configure SSL

import truststore
truststore.inject_into_ssl()

import requests
import json
import pandas as pd
import sys

print("Python:", sys.executable)
print("Requests:", requests.__version__)
print("Pandas:", pd.__version__)
print("SSL certificate: Windows trust store")
print("Import libraries สำเร็จ")

Python: c:\Users\ratchanon\Toolbox\pm25-explorer-0446\.venv\Scripts\python.exe
Requests: 2.34.2
Pandas: 2.3.3
SSL certificate: Windows trust store
Import libraries สำเร็จ


In [2]:
# Cell 2: Test Air4Thai connection

url = "https://air4thai.pcd.go.th/services/getNewAQI_JSON.php"

try:
    response = requests.get(url, timeout=30)

    print("HTTP Status:", response.status_code)
    print("Final URL:", response.url)
    print("Content-Type:", response.headers.get("Content-Type"))

except requests.exceptions.SSLError as e:
    print("SSL Error:", e)

except requests.exceptions.RequestException as e:
    print("Connection Error:", e)


# Cell 3: Fetch data from Air4Thai API

url = "https://air4thai.pcd.go.th/services/getNewAQI_JSON.php"

headers = {
    "User-Agent": "pm25-explorer/0.1 (student project)"
}

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

response.raise_for_status()

data = response.json()

stations = data["stations"]

print("HTTP Status:", response.status_code)
print("จำนวนสถานีทั้งหมด:", len(stations))
print("Top-level keys:", list(data.keys()))


HTTP Status: 200
Final URL: https://air4thai.pcd.go.th/services/getNewAQI_JSON.php
Content-Type: application/json; charset=UTF-8
HTTP Status: 200
จำนวนสถานีทั้งหมด: 181
Top-level keys: ['stations']


In [3]:
# Cell 3: Test general HTTPS connection

try:
    response = requests.get(
        "https://www.google.com",
        timeout=20
    )

    print("Google HTTP Status:", response.status_code)
    print("HTTPS connection: OK")

except requests.exceptions.SSLError as e:
    print("SSL Error:", e)

except requests.exceptions.RequestException as e:
    print("Connection Error:", e)

# Cell 4: Inspect station data structure

sample = stations[0]

print("Station keys:")
print(list(sample.keys()))

print("\nAQILast keys:")
print(list(sample["AQILast"].keys()))

print("\nตัวอย่างข้อมูลสถานี:")
print(json.dumps(sample, ensure_ascii=False, indent=2)[:2000])

Google HTTP Status: 200
HTTPS connection: OK
Station keys:
['stationID', 'nameTH', 'nameEN', 'areaTH', 'areaEN', 'stationType', 'lat', 'long', 'forecast', 'AQILast']

AQILast keys:
['date', 'time', 'PM25', 'PM10', 'O3', 'CO', 'NO2', 'SO2', 'AQI']

ตัวอย่างข้อมูลสถานี:
{
  "stationID": "119t",
  "nameTH": "สวนสาธารณะธารา",
  "nameEN": "Thara Public Park",
  "areaTH": "ต.ปากน้ำ อ.เมือง, กระบี่",
  "areaEN": "Pak Nam Subdistrict, Mueang District, Krabi",
  "stationType": "GROUND",
  "lat": "8.0506237",
  "long": "98.9180489",
  "forecast": [],
  "AQILast": {
    "date": "2026-11-28",
    "time": "22:00",
    "PM25": {
      "color_id": "1",
      "aqi": "15",
      "value": "8.7"
    },
    "PM10": {
      "color_id": "0",
      "aqi": "-1",
      "value": "-1"
    },
    "O3": {
      "color_id": "1",
      "aqi": "25",
      "value": "35"
    },
    "CO": {
      "color_id": "1",
      "aqi": "3",
      "value": "0.60"
    },
    "NO2": {
      "color_id": "1",
      "aqi": "2",
      "

In [4]:
# Cell 5: Create station data table

records = []

for station in stations:
    aqi_last = station.get("AQILast") or {}
    pm25_data = aqi_last.get("PM25") or {}

    record = {
        "station_id": station.get("stationID"),
        "pm25": pm25_data.get("value"),
        "lat": station.get("lat")
    }

    records.append(record)

df = pd.DataFrame(records)

display(df.head(10))

,station_id,pm25,lat
0,119t,8.7,8.0506237
1,02t,21.9,13.732846
2,03t,27.9,13.636514
3,05t,23.1,13.666183
4,12t,22.8,13.70806667
5,50t,28.1,13.729852
6,52t,24.6,13.727622
7,53t,27.1,13.7954248
8,54t,28.4,13.76251667
9,59t,18.1,13.783185


In [5]:
# Cell 6: Basic data inspection

print("จำนวนแถวและคอลัมน์:", df.shape)

print("\nชนิดข้อมูล:")
print(df.dtypes)

print("\nข้อมูลที่หายไป:")
print(df.isnull().sum())

print("\nสถิติพื้นฐาน:")
display(df.describe(include="all"))

จำนวนแถวและคอลัมน์: (181, 3)

ชนิดข้อมูล:
station_id    object
pm25          object
lat           object
dtype: object

ข้อมูลที่หายไป:
station_id    0
pm25          0
lat           0
dtype: int64

สถิติพื้นฐาน:


,station_id,pm25,lat
count,181,181,181
unique,181,125,181
top,119t,22.0,8.0506237
freq,1,4,1


In [6]:
# Cell 7: Convert data types

df["pm25"] = pd.to_numeric(df["pm25"], errors="coerce")
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")

print("ชนิดข้อมูลหลังแปลง:")
print(df.dtypes)

print("\nตรวจสอบ Missing values:")
print(df.isnull().sum())

ชนิดข้อมูลหลังแปลง:
station_id     object
pm25          float64
lat           float64
dtype: object

ตรวจสอบ Missing values:
station_id    0
pm25          0
lat           0
dtype: int64


In [7]:
# Cell 8: Analyze PM2.5 values

print("ค่า PM2.5 ต่ำสุด:", df["pm25"].min())
print("ค่า PM2.5 สูงสุด:", df["pm25"].max())
print("ค่า PM2.5 เฉลี่ย:", round(df["pm25"].mean(), 2))

print("\n5 สถานีที่มีค่า PM2.5 สูงสุด:")
display(
    df.sort_values("pm25", ascending=False).head(5)
)

ค่า PM2.5 ต่ำสุด: -1.0
ค่า PM2.5 สูงสุด: 41.2
ค่า PM2.5 เฉลี่ย: 20.82

5 สถานีที่มีค่า PM2.5 สูงสุด:


,station_id,pm25,lat
96,o69,41.2,13.832076
149,27t,40.2,13.550263
147,84t,39.0,13.428581
89,117t,37.5,15.173475
116,86t,36.2,16.820833


In [8]:
# Cell 9: Handle invalid PM2.5 values

df["pm25"] = df["pm25"].where(df["pm25"] >= 0)

print("จำนวนข้อมูล PM2.5 ที่หายไป:", df["pm25"].isnull().sum())

print("ค่า PM2.5 ต่ำสุด:", df["pm25"].min())
print("ค่า PM2.5 สูงสุด:", df["pm25"].max())
print("ค่า PM2.5 เฉลี่ย:", round(df["pm25"].mean(), 2))

จำนวนข้อมูล PM2.5 ที่หายไป: 1
ค่า PM2.5 ต่ำสุด: 7.6
ค่า PM2.5 สูงสุด: 41.2
ค่า PM2.5 เฉลี่ย: 20.94


In [9]:
# Cell 10: Final data quality check

print("จำนวนแถวและคอลัมน์:", df.shape)

print("\nจำนวน Missing values:")
print(df.isnull().sum())

print("\nข้อมูลตัวอย่างหลังทำความสะอาด:")
display(df.head(10))

print("\nสถิติ PM2.5:")
display(df["pm25"].describe())

จำนวนแถวและคอลัมน์: (181, 3)

จำนวน Missing values:
station_id    0
pm25          1
lat           0
dtype: int64

ข้อมูลตัวอย่างหลังทำความสะอาด:


,station_id,pm25,lat
0,119t,8.7,8.050624
1,02t,21.9,13.732846
2,03t,27.9,13.636514
3,05t,23.1,13.666183
4,12t,22.8,13.708067
5,50t,28.1,13.729852
6,52t,24.6,13.727622
7,53t,27.1,13.795425
8,54t,28.4,13.762517
9,59t,18.1,13.783185



สถิติ PM2.5:


count    180.000000
mean      20.944444
std        6.847005
min        7.600000
25%       16.675000
50%       21.450000
75%       25.125000
max       41.200000
Name: pm25, dtype: float64

## Data Mapping

| Output Column | Source JSON Key | Description |
|---|---|---|
| station_id | stationID | รหัสสถานีตรวจวัดคุณภาพอากาศ |
| pm25 | AQILast → PM25 → value | ค่าความเข้มข้น PM2.5 |
| lat | lat | ละติจูดของสถานีตรวจวัด |